In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
files = sorted(glob.glob('/home/ulyanov/data/stereo/2025-09-24/hmi/*'))
files

['/home/ulyanov/data/stereo/2025-09-24/hmi/hmi.M_45s.20250924_062015_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/stereo/2025-09-24/hmi/hmi.V_45s.20250924_062015_TAI.2.Dopplergram.fits']

In [13]:
file = files[1]

with fits.open(file) as hdul:
    header = hdul[1].header
    data = hdul[1].data

data = rebin(data, 8, update_header=header)

In [14]:
view = View.from_header(header)
view_ = view.update(crota=180, increment=True)

In [15]:
V = (view.velocity(mu_thr=0., cbs=False) - view_.velocity(mu_thr=0., cbs=False)) / 2 / 1000
V -= np.nanmean(V)

plt.figure(figsize=(10,10))
plt.imshow(V, 'seismic', vmin=-2, vmax=2)

plt.tight_layout()

In [16]:
V_ = (data.copy() - view_.reproject(data, view)) / 2 / 1000

plt.figure(figsize=(10,10))
plt.imshow(V_, 'seismic', vmin=-2, vmax=2)

plt.tight_layout()

In [17]:
t = ~np.isnan(V)
x, y = V[t], V_[t]
t = np.where(np.abs(y - x) < 0.5)
x, y = x[t], y[t]

k = np.nanmean(x * y) / np.nanmean(x ** 2)
print(k)

plt.figure(figsize=(10,10))
plt.plot([-2,2], [-2,2], color='black', lw=0.5)
plt.scatter(V, V_, s=0.01)
plt.plot([-2,2], [-2 * k, 2 * k], '--', color='black', lw=1)


plt.xlim(-2,2)
plt.ylim(-2,2)

plt.tight_layout()

0.9708676621500344


In [18]:
V = view.velocity(mu_thr=0., cbs=False) / 1000
V -= np.nanmean(V)

V_ = data.copy() / 1000
V_[np.isnan(V)] = np.nan
V_ -= np.nanmean(V_)

In [24]:
mu = view.mu()
U = V_ - V * k

t = np.where(mu > 0.1)
p = np.polyfit(mu[t], U[t], 3)

dx = 0.1
x = np.arange(dx / 2, 1, dx)
y = []
for x_ in x:
    t = np.where(np.abs(mu - x_) < dx / 2)
    y += [np.nanmean(U[t])]
y = np.array(y)

plt.figure(figsize=(10,10))
plt.scatter(mu, U, s=0.03)
plt.plot(x, y, '--', color='black')

plt.title('Convective blueshift')
plt.xlabel(r'$\mu$')
plt.ylabel('Doppler velocity, km/s')

plt.xlim(0,1)
plt.ylim(-0.6,0.6)

plt.gca().invert_xaxis()
plt.tight_layout()

In [25]:
U_ = np.polyval(p, mu)

plt.figure(figsize=(10,10))
plt.imshow(U - U_, 'seismic', vmin=-1, vmax=1)
plt.tight_layout()